In [2]:
import os, json, time, requests
from sys import excepthook

import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from dotenv import load_dotenv

# Load API key from .env
load_dotenv()
API_KEY = os.getenv("FMP_API_KEY")
print(f"Key loaded: {API_KEY[:6]}...")

# Building an absolute path to the database that works on all Windows setups
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
DB_PATH  = os.path.join(BASE_DIR, "data", "raw", "financials.db")

# Creating the data/raw folder if it doesn't exist yet
os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

# Connect to SQLite
engine = create_engine(f"sqlite:///{DB_PATH}")
with engine.connect() as c:
    print("Database connected.")

print(f"DB location: {DB_PATH}")

Key loaded: AVZIRV...
Database connected.
DB location: C:\Users\user\OneDrive\Desktop\Health-engine\financial-health-engine\data\raw\financials.db


In [3]:
TICKERS = [
    # Banking and Financial Services
    "FRS.JO",               #FirstRand
    "SBK.JO",   # Standard Bank
    "ABG.JO",   # Absa Group
    "NED.JO",   # Nedbank
    "CPI.JO",   # Capitec
    "SLM.JO",   # Sanlam
    "DSY.JO",   # Discovery
    "OMU.JO",   # Old Mutual
    # Retail & Consumer
    "SHP.JO",   # Shoprite
    "PIK.JO",   # Pick n Pay
    "WHL.JO",   # Woolworths
    "TBS.JO",   # Tiger Brands
    # Telecoms
    "MTN.JO",   # MTN Group
    "VOD.JO",   # Vodacom
    "NPN.JO",   # Naspers
    # Mining & Resources
    "AGL.JO",   # Anglo American
    "SOL.JO",   # Sasol
    "ANG.JO",   # AngloGold Ashanti
    "GFI.JO",   # Gold Fields
    "IMP.JO",   # Impala Platinum
    # Diversified / Industrial
    "BVT.JO",   # Bidvest
    "REM.JO",   # Remgro
    "CFR.JO",   # Richemont
]

SECTORS = {
    "FSR.JO": "Banking",  "SBK.JO": "Banking",  "ABG.JO": "Banking",
    "NED.JO": "Banking",  "CPI.JO": "Banking",
    "SLM.JO": "Insurance","DSY.JO": "Insurance", "OMU.JO": "Insurance",
    "SHP.JO": "Retail",   "PIK.JO": "Retail",    "WHL.JO": "Retail",
    "TBS.JO": "FMCG",
    "MTN.JO": "Telecoms", "VOD.JO": "Telecoms",  "NPN.JO": "Tech",
    "AGL.JO": "Mining",   "SOL.JO": "Energy",    "ANG.JO": "Mining",
    "GFI.JO": "Mining",   "IMP.JO": "Mining",
    "BVT.JO": "Industrial","REM.JO": "Holdings", "CFR.JO": "Luxury",
}

NAMES = {
    "FSR.JO": "FirstRand",     "SBK.JO": "Standard Bank", "ABG.JO": "Absa Group",
    "NED.JO": "Nedbank",       "CPI.JO": "Capitec",       "SLM.JO": "Sanlam",
    "DSY.JO": "Discovery",     "OMU.JO": "Old Mutual",    "SHP.JO": "Shoprite",
    "PIK.JO": "Pick n Pay",    "WHL.JO": "Woolworths",    "TBS.JO": "Tiger Brands",
    "MTN.JO": "MTN Group",     "VOD.JO": "Vodacom",       "NPN.JO": "Naspers",
    "AGL.JO": "Anglo American","SOL.JO": "Sasol",         "ANG.JO": "AngloGold",
    "GFI.JO": "Gold Fields",   "IMP.JO": "Implats",       "BVT.JO": "Bidvest",
    "REM.JO": "Remgro",        "CFR.JO": "Richemont",
}

print(f"Universe: {len(TICKERS)} JSE companies across {len(set(SECTORS.values()))} sectors")


Universe: 23 JSE companies across 11 sectors


In [6]:
import yfinance as yf

def fetch_financials(ticker):
    print(f"  Fetching {NAMES.get(ticker, ticker)}...", end=" ")
    try:
        stock = yf.Ticker(ticker)

        income    = stock.financials.T.reset_index()
        balance   = stock.balance_sheet.T.reset_index()
        cashflow  = stock.cashflow.T.reset_index()

        for df in [income, balance, cashflow]:
            df.rename(columns={"index": "date"}, inplace=True)
            df["ticker"]       = ticker
            df["sector"]       = SECTORS.get(ticker, "Other")
            df["company_name"] = NAMES.get(ticker, ticker)
            df["date"]         = pd.to_datetime(df["date"])

        print("OK")
        return income, balance, cashflow

    except Exception as e:
        print(f"ERROR: {e}")
        return None, None, None


# Quick test with Shoprite
print("Testing yfinance with Shoprite (SHP.JO)...")
inc, bal, cf = fetch_financials("SHP.JO")

if inc is not None:
    print(f"\nSuccess! Got {len(inc)} years of data")
    print(inc[["date", "ticker", "Total Revenue", "Net Income"]].head())
else:
    print("Something went wrong")

Testing yfinance with Shoprite (SHP.JO)...
  Fetching Shoprite... OK

Success! Got 5 years of data
        date  ticker  Total Revenue    Net Income
0 2025-06-30  SHP.JO   2.527010e+11  7.585000e+09
1 2024-06-30  SHP.JO   2.320880e+11  6.248000e+09
2 2023-06-30  SHP.JO   2.149560e+11  5.886000e+09
3 2022-06-30  SHP.JO   1.838680e+11  5.711000e+09
4 2021-06-30  SHP.JO            NaN           NaN


In [5]:
# Fetching the financials from all 23 Companies.
all_income    = []
all_balance   = []
all_cashflow  = []
failed        = []

print(f"Fetching data for all {len(TICKERS)} JSE companies...\n")

for i, ticker in enumerate(TICKERS, 1):
    print(f"[{i:02}/{len(TICKERS)}]", end=" ")
    inc, bal, cf = fetch_financials(ticker)

    if inc is not None:
        all_income.append(inc)
        all_balance.append(bal)
        all_cashflow.append(cf)
    else:
        failed.append(ticker)

    time.sleep(0.5)  # Just to be polite to the API

# Combining all companies into single DataFrames
df_income    = pd.concat(all_income,   ignore_index=True) if all_income   else None
df_balance   = pd.concat(all_balance,  ignore_index=True) if all_balance  else None
df_cashflow  = pd.concat(all_cashflow, ignore_index=True) if all_cashflow else None

print(f"\n✓ Income:    {len(df_income)} rows")
print(f"✓ Balance:   {len(df_balance)} rows")
print(f"✓ Cashflow:  {len(df_cashflow)} rows")

if failed:
    print(f"\nFailed tickers: {failed}")
else:
    print("\nAll 23 companies fetched successfully!")

Fetching data for all 23 JSE companies...

[01/23]   Fetching FRS.JO... OK
[02/23]   Fetching Standard Bank... OK
[03/23]   Fetching Absa Group... OK
[04/23]   Fetching Nedbank... OK
[05/23]   Fetching Capitec... OK
[06/23]   Fetching Sanlam... OK
[07/23]   Fetching Discovery... OK
[08/23]   Fetching Old Mutual... OK
[09/23]   Fetching Shoprite... OK
[10/23]   Fetching Pick n Pay... OK
[11/23]   Fetching Woolworths... OK
[12/23]   Fetching Tiger Brands... OK
[13/23]   Fetching MTN Group... OK
[14/23]   Fetching Vodacom... OK
[15/23]   Fetching Naspers... OK
[16/23]   Fetching Anglo American... OK
[17/23]   Fetching Sasol... OK
[18/23]   Fetching AngloGold... OK
[19/23]   Fetching Gold Fields... OK
[20/23]   Fetching Implats... OK
[21/23]   Fetching Bidvest... OK
[22/23]   Fetching Remgro... OK
[23/23]   Fetching Richemont... OK

✓ Income:    101 rows
✓ Balance:   101 rows
✓ Cashflow:  102 rows

All 23 companies fetched successfully!


In [7]:
# Saving to SQLite.
# Save all three statements to SQLite
df_income.to_sql("income",   engine, if_exists="replace", index=False)
df_balance.to_sql("balance",  engine, if_exists="replace", index=False)
df_cashflow.to_sql("cashflow", engine, if_exists="replace", index=False)

print("Saved to database!")
print(f"\nVerification:")
print(f"  Income:   {pd.read_sql('SELECT COUNT(*) as rows FROM income',   engine).iloc[0,0]} rows")
print(f"  Balance:  {pd.read_sql('SELECT COUNT(*) as rows FROM balance',  engine).iloc[0,0]} rows")
print(f"  Cashflow: {pd.read_sql('SELECT COUNT(*) as rows FROM cashflow', engine).iloc[0,0]} rows")

# Preview — top 5 companies by latest revenue
print("\nTop 5 by Revenue (latest year):")
preview = pd.read_sql("""
                      SELECT company_name, date, "Total Revenue", "Net Income"
                      FROM income
                      ORDER BY "Total Revenue" DESC
                          LIMIT 5
                      """, engine)
print(preview)

Saved to database!

Verification:
  Income:   101 rows
  Balance:  101 rows
  Cashflow: 102 rows

Top 5 by Revenue (latest year):
  company_name                        date  Total Revenue    Net Income
0        Sasol  2023-06-30 00:00:00.000000   2.896960e+11  8.799000e+09
1        Sasol  2024-06-30 00:00:00.000000   2.751110e+11 -4.427100e+10
2        Sasol  2022-06-30 00:00:00.000000   2.727460e+11  3.895600e+10
3     Shoprite  2025-06-30 00:00:00.000000   2.527010e+11  7.585000e+09
4        Sasol  2025-06-30 00:00:00.000000   2.490960e+11  6.767000e+09
